# LangGraph

LangGraph is a way to build AI workflow as a graph (nodes + edges) instead of straight line.

If LangChain is a pipeline, then LangGraph is a flowchart with memory, loops and decision

---
Why LangGraph exists

**Normal LLM chains:**

Input --> LLM --> Output

**Need of a real time application (AI application):**
* Loops (retry, think again)
* Decision making (if/else)
* Memory (state)
* Multi agents to talk to each other
* Control statement (stop, continue, route)

==> LangGraph can solve/handle all this cleanly


---
# Core concepts

LangGraph = State + Nodes + Edges + Control

---
# State (The Memory)

State is a shared memory that flows through the graph

**Example:**
```pytnon
state = {
    "question": "What is LangGraph?",
    "answer": ""
}
```

---
# Nodes (The workers)

Nodes are functions that do work

Each node:
* takes the state
* does something (llm call, logic, tool)
* returns update state

**Example Node:**
```python
def generate_answer(state):
    return{
        "answer":"LangGraph is a graph-based workflow for LLMS"
    }
```

**PS:**

Node = one step in thinking

---
# Edges (The flow)

Edges define who runs after whom

**Types of Edges:**
1. Normal edge --> go the next node
2. Conditional edge --> choose the next node
3. Looping edge --> go back to previous node
4. End edge --> stop

**Simple Edge:**
START --> generate_answer --> END

---
# Control (Graph) - putting all of it together

We are going define following things:
1. State schema
2. Nodes
3. Edges
4. Entry point
5. End point

---
# Simple LangGraph example

In [ ]:
# installing the langgraph package
!pip install langgraph

In [ ]:
# Define the state
from typing import TypedDict

class State(TypedDict):
    number: int
    result: int

# Define the node
def square_node(state: State) -> State:
    """
    Node Logic:
    - reading the number from the state
    - square it
    - return as result
    """
    num = state["number"]
    return {
        "result": num * num
    }

# Build the Graph
from langgraph.graph import StateGraph, END

graph = StateGraph(State)

# Add the node
graph.add_node("square", square_node)

# Entry point
graph.set_entry_point("square")

# Edge (where to go next)
graph.add_edge("square", END)

# Compile of the graph
app = graph.compile()

# Run the graph
output = app.invoke(
    {
        "number": 25
    }
)

print("Result: ", output["result"])



# Memory Management

In [22]:
from typing import TypedDict

# State definition
class State(TypedDict): 
    number: int
    total: int # use this as memory for conversation

#  node definition
def add_node(state: State) -> State:
    """
    Reads:
        number (input)
        total  (memory)
    Updates:
        total
    """
    return {
        "total": state.get("total", 0) + state["number"]
    }

# building the graph
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

graph = StateGraph(State)

graph.add_node("add", add_node)
graph.set_entry_point("add")
graph.add_edge("add", END)

# Adding the memory
checkpointer = MemorySaver()

app = graph.compile(checkpointer=checkpointer)

# persistent memory
config = {"configurable": {"thread_id": "user-1"}}

print("After adding 5 --> ", app.invoke({"number": 5}, config)["total"])
print("After adding 3 --> ",app.invoke({"number": 3}, config)["total"])
print("After adding 20 --> ",app.invoke({"number": 20}, config)["total"])



Before execution, total =  0
After adding 5 -->  5
After adding 3 -->  8
This is with thread id-2 After adding 3 -->  3
After adding 20 -->  28


---
# Sample LLM call

In [5]:
from langgraph.graph import StateGraph, MessagesState, START, END

def mock_llm(state: MessagesState):
    return {"messages": [{"role": "ai", "content": "hello world"}]}

graph = StateGraph(MessagesState)
graph.add_node(mock_llm)
graph.add_edge(START, "mock_llm")
graph.add_edge("mock_llm", END)

graph = graph.compile()

res = graph.invoke({"messages": [{"role": "user", "content":"hi from user!!!"}]})

In [6]:
print(res)

{'messages': [HumanMessage(content='hi from user!!!', additional_kwargs={}, response_metadata={}, id='22592d66-de18-4581-aef6-2d5bb3dcdbfd'), AIMessage(content='hello world', additional_kwargs={}, response_metadata={}, id='0a58067a-0274-4725-ad7c-4d32e9ff2176', tool_calls=[], invalid_tool_calls=[])]}


In [8]:
print(res['messages'])

[HumanMessage(content='hi from user!!!', additional_kwargs={}, response_metadata={}, id='22592d66-de18-4581-aef6-2d5bb3dcdbfd'), AIMessage(content='hello world', additional_kwargs={}, response_metadata={}, id='0a58067a-0274-4725-ad7c-4d32e9ff2176', tool_calls=[], invalid_tool_calls=[])]


In [9]:
type(res['messages'])

list

In [14]:
for msg in res["messages"]:
    print (msg)

content='hi from user!!!' additional_kwargs={} response_metadata={} id='22592d66-de18-4581-aef6-2d5bb3dcdbfd'
content='hello world' additional_kwargs={} response_metadata={} id='0a58067a-0274-4725-ad7c-4d32e9ff2176' tool_calls=[] invalid_tool_calls=[]


---
# Multi agent LangGraph Code

Types of Multi agents state
1. With Single shared State
2. Hybrid state

---
### 1. With Single Shared State

LangGraph = State(shared state) + Nodes (node1, node2, node3, etc..) + edges + contrrol

**Flow:**
State --> Nodes --> Edges --> Compile --> Invoke

In [17]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# define the state
class State(TypedDict):
    message: str

# agent-1
def agent_one(state: State):
    print("Agent one is running...")
    return {"message": "Hello from Agent one"}

# agent-2
def agent_two(state: State):
    print("Agent two is running...")
    return {"message": f"Agent two received following message from previous agent: {state['message']}"}

# build the graph
graph = StateGraph(State)

graph.add_node("agent_one", agent_one)
graph.add_node("agent_two", agent_two)

graph.set_entry_point("agent_one")
graph.add_edge("agent_one", "agent_two")
graph.add_edge("agent_two", END)

# compile the graph
app = graph.compile()

# run the graph
res = app.invoke({"message": ""})
print(res)

Agent one is running...
Agent two is running...
{'message': 'Agent two received following message from previous agent: Hello from Agent one'}


---
### Hybrid State (Most realistic)

LangGraph = State(Global State and Local State) + Nodes (node1, node2, node3, etc..) + edges + control

* Global State ==> shared by every agent
* Local State ==> private to each agent
* Observation ==> agent sees only what it needs

**Example Scenario:**

Two Agent demo:

**1. Planner Agent**
   ```
   Sees: goal
   Produces: plan
   ```
**2. Executor Agent**
   ```
   Sees: plan
   Produces: result
   ```

In [24]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END

# hybrid state (global)
class GlobalState(TypedDict):
    goal: str
    plan: Optional[str]
    result: Optional[str]

# Agent-1: Planner Agent (local state inside it)
def planner_agent(state: GlobalState):
    # partial observation
    goal = state["goal"]

    # local private reasoning (NOT Shared)
    local_thoughts = f"I need steps to achieve: {goal}"

    plan = f"\n step-1: Analyze the goal \n step-2: Execute"

    print("Planner with local thoughts:", local_thoughts)

    # only return what should be shared to global state
    return {"plan": plan}


# Agent-2: Executor agent (partial view)
def executor_agent(state: GlobalState):
    # sees the plan and not the goal
    plan = state["plan"]

    local_execution_log = f"Executing plan: {plan}"

    result = "Task completely successfully."

    print("Executor local log:", local_execution_log)
    return {"result": result}


# build the graph
graph = StateGraph(GlobalState)

graph.add_node("planner", planner_agent)
graph.add_node("executor", executor_agent)

graph.set_entry_point("planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", END)

# compile the graph
app = graph.compile()

# run the graph

result = app.invoke(
    {
        "goal": "Build a multi-agent system",
        "plan": None,
        "result": None
    }
)

print(result)

Planner with local thoughts: I need steps to achieve: Build a multi-agent system
Executor local log: Executing plan: 
 step-1: Analyze the goal 
 step-2: Execute
{'goal': 'Build a multi-agent system', 'plan': '\n step-1: Analyze the goal \n step-2: Execute', 'result': 'Task completely successfully.'}
